# LMDI contribution analysis

Computes transport-distance decomposition contributions from synthetic panel samples. It uses `src/lmdi.py` and expects generated panel data and prepared feature lists. Outputs include mode-level frequency, mode-share, global-length, and relative-length effects.

In [ ]:
# Load necessary libraries
import pandas as pd
import numpy as np
import os
import warnings
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid", {'axes.grid': False})
warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)

import sys
sys.path.append(r'..\src')
import lmdi as ld

In [ ]:
# Load generated data
path = '../'
pop_size = 4
model = 'best_cvae_model'
var = 100

df_samples_con = pd.read_parquet(path + f"data\TUmodeldata_panel_ally_{pop_size}.parquet")  # Load as DataFrame

# Load feature lists
with open(path +"data/pref_cat.txt", "r") as f:
    cat_features = f.read().splitlines()
with open(path +"data/pref_num.txt", "r") as f:
    num_features = f.read().splitlines()
with open(path +"data/con_num.txt", "r") as f:
    num_cond_features = f.read().splitlines()
with open(path +"data/con_cat.txt", "r") as f:
    cat_cond_features = f.read().splitlines()

# List of trips columns and average length columns
trips_columns = cat_features[1:]
avg_length_columns = num_features

### Functions

In [ ]:
def read_parquet_file_pandas(file, df_samples_con):
    df = pd.read_parquet(file)  # Read as a pandas DataFrame

    # Calculate mileage per mode
    trcol = ['trips_mode_1', 'trips_mode_2', 'trips_mode_3', 'trips_mode_4', 'trips_mode_5', 'trips_mode_6', 'trips_mode_7']
    lencol = ['lengths_mode_1', 'lengths_mode_2', 'lengths_mode_3', 'lengths_mode_4', 'lengths_mode_5', 'lengths_mode_6', 'lengths_mode_7']
    for i in range(len(trips_columns)):
        mileage_column = f'mileage_mode_{i+1}'  # Create a new column name for mileage
        df[mileage_column] = df[trcol[i]] * df[lencol[i]]

    # Reset the index to ensure known divisions
    df = df.reset_index(drop=True)
    df_samples_con = df_samples_con.reset_index(drop=True)

    # Concatenate the two DataFrames
    df_combined = pd.concat([df, df_samples_con], axis=1)

    return df_combined

def calc_subst(path, df_samples_con, segment_cols=None, year_col='DiaryYear', 
               year1=2015, year2=2024, maxfiles=100):
    """
    Calculate LMDI decomposition for modal substitution analysis.
    Processes multiple samples and returns mean and std for each variable.
    
    Parameters:
    -----------
    path : str
        Base path to data directory
    df_samples_con : DataFrame
        Conditioning data to merge with samples
    segment_cols : list of str or None
        Column names to segment by (e.g., ['AgeGroup'], ['AgeGroup', 'Gender'])
        If None, calculates overall decomposition
    year_col : str
        Column name containing year information
    year1, year2 : int
        Years to compare (state Î» and Î»')
    maxfiles : int
        Maximum number of parquet files to process
    
    Returns:
    --------
    Tuple of (df_summary, df_all_samples)
        - df_summary: Mean and std values across all samples
        - df_all_samples: Complete raw results from all samples
    """
    
    mode_mapping = {
        1: 'Walk', 2: 'Bike', 3: 'E-bike',
        4: 'Car', 5: 'Car pas.', 6: 'Pub.tr.', 7: 'Moped'
    }
    mode_names = list(mode_mapping.values())
    decomposer = ld.TransportDecomposition(modes=mode_names)
    
    directory_path = os.path.join(path, f"results/samples_{pop_size}_{model}/")
    parquet_files = [
        os.path.join(directory_path, f)
        for f in os.listdir(directory_path)
        if f.endswith(".parquet")
    ][:maxfiles]

    print(f"\nProcessing {len(parquet_files)} sample files...")
    
    trips_columns   = [f'trips_mode_{i}'   for i in range(1, 8)]
    mileage_columns = [f'mileage_mode_{i}' for i in range(1, 8)]
    
    purpose_map = {
        0: 'Leisure/Other', 1: 'Leisure/Other', 2: 'Leisure/Other',
        3: 'Leisure/Other', 4: 'Work/Education', 5: 'Leisure/Other',
        6: 'Work/Education', 7: 'Leisure/Other'
    }

    all_results = []

    for file_idx, file in enumerate(parquet_files):
        if (file_idx + 1) % 10 == 0:
            print(f"  Processing file {file_idx + 1}/{len(parquet_files)}...")
        
        ddf_new = read_parquet_file_pandas(file, df_samples_con)
        ddf_new['purposes'] = ddf_new['purposes'].map(purpose_map)
       
        segment_groups = [(None, ddf_new)] if segment_cols is None else list(ddf_new.groupby(segment_cols))

        for segment_key, segment_data in segment_groups:
            state1_df = segment_data[segment_data[year_col] == year1]
            state2_df = segment_data[segment_data[year_col] == year2]
            
            if len(state1_df) == 0 or len(state2_df) == 0:
                continue

            n1, n2 = len(state1_df), len(state2_df)
            if n1 != n2:
                print(f"  Warning: Different population sizes for {os.path.basename(file)}, segment {segment_key}: n1={n1}, n2={n2}")
            
            Q1 = state1_df[trips_columns].sum().values
            D1 = state1_df[mileage_columns].sum().values
            L1 = D1 / np.where(Q1 > 0, Q1, 1)
            state1 = {
                'trips':       {mode_mapping[i+1]: Q1[i] for i in range(len(Q1))},
                'avg_lengths': {mode_mapping[i+1]: L1[i] for i in range(len(L1))}
            }
            
            Q2 = state2_df[trips_columns].sum().values
            D2 = state2_df[mileage_columns].sum().values
            L2 = D2 / np.where(Q2 > 0, Q2, 1)
            state2 = {
                'trips':       {mode_mapping[i+1]: Q2[i] for i in range(len(Q2))},
                'avg_lengths': {mode_mapping[i+1]: L2[i] for i in range(len(L2))}
            }

            if sum(state1['trips'].values()) == 0 or sum(state2['trips'].values()) == 0:
                continue
            
            try:
                df_decomp = decomposer.lmdi_decompose_fixed(state1, state2, n1, n2)
            except Exception as e:
                print(f"Error in decomposition for {os.path.basename(file)}, segment {segment_key}: {e}")
                continue

            # Check df_decomp structure and type
            print(f"  Decomposition result for {os.path.basename(file)}, segment {segment_key}:")
            
            for _, row in df_decomp.iterrows():
                result = {
                    'sample_id': os.path.basename(file).replace('.parquet', ''),
                    'mode':      row['mode'],
                    'Delta_D':      row['distance_change'],
                    'Delta_D_perc': row['per_dist_change'],
                    'Delta_D_F':    row['frequency_effect'],
                    'Delta_D_C':    row['mode_share_effect'],
                    'Delta_D_G':    row['global_length_effect'],
                    'Delta_D_L':    row['relative_length_effect'],
                    'phi_F': row['phi_F'],
                    'phi_C': row['phi_C'],
                    'phi_G': row['phi_G'],
                    'phi_L': row['phi_L'],
                    'decomp_verification': row['verification'],
                    'Q1': row['Q1'], 'Q2': row['Q2'],
                    'D1': row['D1'], 'D2': row['D2'],
                    'L1': row['L1'], 'L2': row['L2'],
                }
                
                if segment_cols is not None:
                    if isinstance(segment_key, tuple):
                        for i, col in enumerate(segment_cols):
                            result[col] = segment_key[i]
                    else:
                        result[segment_cols[0]] = segment_key
                
                all_results.append(result)

    df_all_samples = pd.DataFrame(all_results)
    
    if len(df_all_samples) == 0:
        print("Warning: No results generated. Check your data and filters.")
        return None, pd.DataFrame()
    
    print(f"\nTotal results collected: {len(df_all_samples)} records from {df_all_samples['sample_id'].nunique()} samples")
    
    # Verification: should equal 1
    df_all_samples['decomp_sum'] = (
        df_all_samples['phi_F'] + df_all_samples['phi_C'] +
        df_all_samples['phi_G'] + df_all_samples['phi_L']
    )
    
    group_cols = ['mode'] if segment_cols is None else segment_cols + ['mode']
    
    numeric_cols = [
        'Delta_D', 'Delta_D_perc', 'Delta_D_F', 'Delta_D_C', 'Delta_D_G', 'Delta_D_L',
        'phi_F', 'phi_C', 'phi_G', 'phi_L',
        'decomp_sum', 'decomp_verification',
        'Q1', 'Q2', 'D1', 'D2', 'L1', 'L2'
    ]
    
    df_mean = df_all_samples.groupby(group_cols)[numeric_cols].mean().reset_index()
    df_mean['n_samples'] = df_all_samples.groupby(group_cols).size().values
    
    df_std = df_all_samples.groupby(group_cols)[numeric_cols].std().reset_index()
    df_std = df_std.rename(columns={col: f'{col}_std' for col in numeric_cols})
    
    for col in ['Delta_D', 'Delta_D_perc', 'phi_F', 'phi_C', 'phi_G', 'phi_L']:
        df_mean[f'{col}_cv'] = np.where(
            np.abs(df_mean[col]) > 1e-6,
            df_std[f'{col}_std'] / np.abs(df_mean[col]),
            np.nan
        )
    
    df_summary = df_mean.merge(df_std, on=group_cols, how='left')
    
    print(f"\n{'='*80}")
    print(f"LMDI DECOMPOSITION RESULTS SUMMARY")
    print(f"Total samples: {df_all_samples['sample_id'].nunique()} | Records: {len(df_all_samples)}")
    if segment_cols:
        print(f"Segmentation: {segment_cols}")
    print(f"Gainer modes: {df_mean[df_mean['Delta_D'] > 0]['mode'].unique()}")
    print(f"Decomp sum â€” mean: {abs(df_mean['decomp_sum']).mean():.6f} | min: {abs(df_mean['decomp_sum']).min():.6f} | max: {abs(df_mean['decomp_sum']).max():.6f}")
    for col in ['Delta_D', 'Delta_D_perc', 'phi_F', 'phi_C', 'phi_G', 'phi_L']:
        cv = df_mean[f'{col}_cv'].mean()
        print(f"  CV {col:15s}: {cv:.4f}")
    print(f"{'='*80}\n")
    
    return df_summary, df_all_samples

In [ ]:
def summarize_decomposition_results(df_summary, segment_cols=None, output_path=None):
    """
    Print summary tables of decomposition results.
    
    Parameters:
    -----------
    df_summary : DataFrame
        Summary results from calc_subst (combined mean and std)
    segment_cols : list of str or None
        Segmentation columns used
    output_path : str or None
        Path to save summary as CSV
    """
    
    effect_cols   = ['phi_F', 'phi_C', 'phi_G', 'phi_L']
    
    print("\n" + "="*80)
    print("EFFECT CONTRIBUTIONS BY MODE (Mean Â± Std)")
    print("="*80)
    
    for mode in df_summary['mode'].unique():
        mode_data = df_summary[df_summary['mode'] == mode]
        print(f"\nMode: {mode}")
        
        if segment_cols:
            for _, row in mode_data.iterrows():
                seg_str = ", ".join([f"{col}={row[col]}" for col in segment_cols])
                print(f"  Segment: {seg_str}")
                for effect in effect_cols:
                    print(f"    {effect}: {row[effect]:.4f} Â± {row[f'{effect}_std']:.4f} (CV={row.get(f'{effect}_cv', float('nan')):.4f})")
        else:
            for effect in effect_cols:
                mean_val = mode_data[effect].iloc[0]
                std_val  = mode_data[f'{effect}_std'].iloc[0]
                cv_val   = mode_data[f'{effect}_cv'].iloc[0] if f'{effect}_cv' in mode_data.columns else float('nan')
                print(f"  {effect}: {mean_val:.4f} Â± {std_val:.4f} (CV={cv_val:.4f})")
    
    print("\n" + "="*80)
    print("DISTANCE CHANGES BY MODE (Mean Â± Std)")
    print("="*80)
    
    for mode in df_summary['mode'].unique():
        mode_data = df_summary[df_summary['mode'] == mode]
        print(f"\nMode: {mode}")
        
        if segment_cols:
            for _, row in mode_data.iterrows():
                seg_str = ", ".join([f"{col}={row[col]}" for col in segment_cols])
                print(f"  {seg_str}: {row['Delta_D']:.2f} Â± {row['Delta_D_std']:.2f} km")
                print(f"    Per person: {row['Delta_D_perc']:.2f} Â± {row['Delta_D_perc_std']:.2f} km/person")
        else:
            print(f"  {mode_data['Delta_D'].iloc[0]:.2f} Â± {mode_data['Delta_D_std'].iloc[0]:.2f} km")
            print(f"  Per person: {mode_data['Delta_D_perc'].iloc[0]:.2f} Â± {mode_data['Delta_D_perc_std'].iloc[0]:.2f} km/person")
    
    print("\n" + "="*80 + "\n")
    
    if output_path:
        df_summary.to_csv(output_path, index=False)
        print(f"Summary saved to: {output_path}")


def plot_decomposition_results(df_summary, df_all_samples, segment_cols=None,
                               output_dir=None, show_uncertainty=True):
    """
    Create visualizations of decomposition results with uncertainty.
    
    Parameters:
    -----------
    df_summary : DataFrame
        Summary statistics from calc_subst
    df_all_samples : DataFrame
        All individual sample results from calc_subst
    segment_cols : list of str or None
        Segmentation columns
    output_dir : str or None
        Directory to save plots
    show_uncertainty : bool
        Whether to show error bars
    """
   
    modes         = ['Walk', 'Bike', 'E-bike', 'Car', 'Car pas.', 'Pub.tr.']
    effects       = ['phi_F', 'phi_C', 'phi_G', 'phi_L']
    effect_labels = ['Frequency', 'Mode Share', 'Global Length', 'Relative Length']
    colors        = ['#1f77b4', '#6baed6', '#fdae6b', '#ff7f0e']
    
    # --- Plot 1: Effect contributions + distance changes ---
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # Common ordering
    y = np.arange(len(modes))
    bar_h = 0.18

    # -------------------
    # Decomposition effects
    # -------------------
    ax = axes[1]

    for i, (effect, label, color) in enumerate(zip(effects, effect_labels, colors)):
        means = [df_summary[df_summary['mode'] == m][effect].iloc[0] for m in modes]
        stds  = [df_summary[df_summary['mode'] == m][f'{effect}_std'].iloc[0] for m in modes]

        ax.barh(
            y + (i - (len(effects)-1)/2) * bar_h,
            means,
            height=bar_h,
            label=label,
            color=color,
            xerr=stds if show_uncertainty else None,
            capsize=3,
            alpha=0.8,
            edgecolor='black'
        )

    ax.set_yticks([])      
    ax.set_xlabel('Contribution ($\\phi$)', fontsize=12)
    # ax.set_ylabel('Mode', fontsize=12)
    ax.axvline(0, color='black', linewidth=0.8, linestyle='--', alpha=0.5)
    ax.grid(axis='x', alpha=0.3)
    ax.legend(fontsize=10)

    # -------------------
    # Mileage change
    # -------------------
    ax = axes[0]

    means = np.array([
        df_summary[df_summary['mode'] == m]['Delta_D_perc'].iloc[0]
        for m in modes
    ])

    stds = np.array([
        df_summary[df_summary['mode'] == m]['Delta_D_perc_std'].iloc[0]
        for m in modes
    ])

    ax.barh(
        y,
        means,
        xerr=stds if show_uncertainty else None,
        capsize=3,
        color='#1f77b4',
        alpha=0.7,
        edgecolor='black'
    )

    ax.set_yticks(y)
    ax.set_yticklabels(modes)
    ax.set_xlabel('Mileage Change (km/person)', fontsize=12)
    ax.set_ylabel('Mode', fontsize=12)
    ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
    ax.grid(axis='x', alpha=0.3)

    plt.tight_layout()
    _save_or_show(fig, output_dir, 'decomposition_summary.png')


    # --- Plot 2: Distribution of effects (violin plots) ---
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    for i, (effect, label) in enumerate(zip(effects, effect_labels)):
        ax = axes.flatten()[i]
        plot_data   = [df_all_samples[df_all_samples['mode'] == m][effect].values for m in modes]
        plot_data   = [d for d in plot_data if len(d) > 0]
        plot_labels = [m for m, d in zip(modes, [df_all_samples[df_all_samples['mode'] == m][effect].values for m in modes]) if len(d) > 0]
        
        if plot_data:
            ax.violinplot(plot_data, positions=range(len(plot_labels)), showmeans=True, showmedians=True)
            ax.set_xticks(range(len(plot_labels)))
            ax.set_xticklabels(plot_labels)
            ax.set_ylabel(f'{label} (Ï†)', fontsize=11)
            ax.set_title(f'Distribution of {label} Effect', fontsize=12)
            ax.axhline(0, color='black', linewidth=0.8, linestyle='--', alpha=0.3)
            ax.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    _save_or_show(fig, output_dir, 'decomposition_distributions.png')

    # --- Plot 3: Coefficient of variation ---
    fig, ax = plt.subplots(figsize=(12, 6))
    cv_data   = [df_summary[f'{e}_cv'].dropna().values for e in effects if f'{e}_cv' in df_summary.columns]
    cv_labels = [e.replace('phi_', '') for e in effects if f'{e}_cv' in df_summary.columns]
    
    if cv_data:
        bp = ax.boxplot(cv_data, labels=cv_labels, patch_artist=True)
        for patch in bp['boxes']:
            patch.set_facecolor('#1f77b4')
            patch.set_alpha(0.7)
        ax.set_ylabel('Coefficient of Variation', fontsize=12)
        ax.set_xlabel('Effect', fontsize=12)
        ax.set_title('Uncertainty in Effect Estimates (CV across samples)', fontsize=14)
        ax.axhline(0.1, color='orange', linewidth=1, linestyle='--', label='10% threshold', alpha=0.7)
        ax.legend()
        ax.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    _save_or_show(fig, output_dir, 'decomposition_uncertainty.png')


def _save_or_show(fig, output_dir, filename):
    """Helper to save or display a figure."""
    if output_dir:
        path = os.path.join(output_dir, filename)
        fig.savefig(path, dpi=300, bbox_inches='tight')
        print(f"Plot saved to: {path}")
    else:
        fig.show()
    plt.close(fig)


def run_full_analysis(path, df_samples_con, segment_cols=None,
                      year1=2015, year2=2024, maxfiles=1000,
                      output_dir='decomposition_results'):
    """
    Run complete decomposition analysis with summary statistics and visualizations.
    
    Parameters:
    -----------
    path : str
        Base path to data
    df_samples_con : DataFrame
        Conditioning data
    segment_cols : list of str or None
        Segmentation columns
    year1, year2 : int
        Years to compare
    maxfiles : int
        Max number of samples to process
    output_dir : str
        Directory to save outputs
    
    Returns:
    --------
    Tuple of (df_summary, df_all_samples)
    """
    os.makedirs(output_dir, exist_ok=True)
    
    print("="*80)
    print("RUNNING LMDI DECOMPOSITION ANALYSIS")
    print("="*80)
    
    df_summary, df_all_samples = calc_subst(
        path=path, df_samples_con=df_samples_con,
        segment_cols=segment_cols, year_col='DiaryYear',
        year1=year1, year2=year2, maxfiles=maxfiles
    )
    
    if df_summary is None:
        print("No results generated.")
        return None, None
    
    summarize_decomposition_results(
        df_summary, segment_cols=segment_cols,
        output_path=os.path.join(output_dir, 'decomposition_summary.csv')
    )
    
    plot_decomposition_results(
        df_summary, df_all_samples,
        segment_cols=segment_cols,
        output_dir=output_dir,
        show_uncertainty=True
    )
    
    # df_all_samples.to_csv(os.path.join(output_dir, 'all_samples_raw.csv'), index=False)
    
    print(f"\nAnalysis complete. Results saved to: {output_dir}")
    print("="*80)
    
    return df_summary, df_all_samples

### Full population

In [ ]:
output_dir='./results/decomposition_overall'

# Run analysis without segmentation
df_summary, df_all_samples = run_full_analysis(
    path=path,
    df_samples_con=df_samples_con,
    segment_cols=None,  # No segmentation
    year1=2016,
    year2=2024,
    maxfiles=101,  # Adjust based on available samples
    output_dir=output_dir
)


In [ ]:
# Generate plots
output_dir='./results/decomposition_overall'
plot_decomposition_results(
    df_summary,
    df_all_samples,
    segment_cols=None,
    output_dir= output_dir,
    show_uncertainty=True
)

## Detailed segments

#### Gender

In [ ]:
# Run analysis without segmentation
df_summary, df_all_samples = run_full_analysis(
    path=path,
    df_samples_con=df_samples_con,
    segment_cols=['RespSex'],  # No segmentation
    year1=2015,
    year2=2024,
    maxfiles=101,  # Adjust based on available samples
)

#### Age

In [ ]:
# Create quartiles
col_to_cut = 'RespAgeCorrect'
quantiles = df_samples_con[col_to_cut].quantile([0.25, 0.5, 0.75])
print(quantiles)
bins= [df_samples_con[col_to_cut].min()-1, quantiles[0.25], quantiles[0.5], quantiles[0.75], df_samples_con[col_to_cut].max()+1]
labels = ['Q1', 'Q2', 'Q3', 'Q4']
df_samples_con[col_to_cut + 'Group'] = pd.cut(df_samples_con[col_to_cut], bins=bins, labels=labels) 

In [ ]:
# Run analysis without segmentation
df_summary, df_all_samples = run_full_analysis(
    path=path,
    df_samples_con=df_samples_con,
    segment_cols=['RespAgeCorrectGroup'],  # No segmentation
    year1=2015,
    year2=2024,
    maxfiles=101,  # Adjust based on available samples
)

#### Income

In [ ]:
# Create income quartiles manulally 0.25 /432.0 /0.50 728.0 /0.75 1027.0
bins = [-np.inf, 432.0, 728.0, 1027.0, np.inf]
labels = ['low', 'medium', 'high', 'very_high']
df_samples_con['IncomeGroup'] = pd.cut(df_samples_con['IncFamily2023'], bins=bins, labels=labels)

# Run analysis
df_summary, df_all_samples = run_full_analysis(
    path=path,
    df_samples_con=df_samples_con,
    segment_cols=['IncomeGroup'],
    year1=2015,
    year2=2024,
    maxfiles=101,  # Adjust based on available samples
)

#### Car ownership

In [ ]:
df_samples_con['HousehNumcarsv2'] = df_samples_con['HousehNumcars'].apply(lambda x: 3 if x > 3 else x)

# Run analysis
df_summary, df_all_samples = run_full_analysis(
    path=path,
    df_samples_con=df_samples_con,
    segment_cols=['HousehNumcarsv2'],
    year1=2015,
    year2=2024,
    maxfiles=101,  # Adjust based on available samples
)

#### Purpose

In [ ]:
# Run analysis
df_summary, df_all_samples = run_full_analysis(
    path=path,
    df_samples_con=df_samples_con,
    segment_cols=['purposes'], 
    year1=2015,
    year2=2024,
    maxfiles=101,  # Adjust based on available samples
)

#### Accessibility to pop car

In [ ]:
# Function to assign quartiles within each year
def assign_quartiles(group):
    # Compute year-specific quartiles
    q = group['access_pop_exp_car'].quantile([0.25, 0.5, 0.75])
    
    # Define bins using these quartiles
    bins = [
        group['access_pop_exp_car'].min() - 1,
        q.loc[0.25],
        q.loc[0.5],
        q.loc[0.75],
        group['access_pop_exp_car'].max() + 1
    ]
    
    # Assign quartile labels
    group['access_pop_exp_carGroup'] = pd.cut(
        group['access_pop_exp_car'], 
        bins=bins, 
        labels=['Q1', 'Q2', 'Q3', 'Q4'],
        include_lowest=True
    )
    
    return group

# Apply per-year
df_samples_con = df_samples_con.groupby('DiaryYear', group_keys=False).apply(assign_quartiles)

# Run analysis
df_summary, df_all_samples = run_full_analysis(
    path=path,
    df_samples_con=df_samples_con,
    segment_cols=['access_pop_exp_carGroup'],
    year1=2015,
    year2=2024,
    maxfiles=101,  # Adjust based on available samples
)